In [ ]:
# Loading Libraries

import copy
import random
from dataclasses import dataclass
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset, TensorDataset
from torchvision import transforms
from pathlib import Path

torch.manual_seed(0)
random.seed(0)
np.random.seed(0)

In [ ]:
@dataclass
class Config:
    epochs: int = 500
    refit_epochs: int = 500
    patience: int = 5
    min_epochs: int = 20
    min_delta: float = 1e-4
    batch_size: int = 2048
    lr: float = 1e-2
    max_depth: int = 2
    device: str | None = None
    augment_train: bool = True
    eval_every: int = 5  # Reduces training overhead

    def __post_init__(self):
        if self.device is None:
            if torch.cuda.is_available():
                self.device = "cuda"
            elif torch.backends.mps.is_available():
                self.device = "mps"
            else:
                self.device = "cpu"
        self.device = torch.device(self.device)

cfg = Config()
print(cfg)

In [ ]:
def make_loader(dataset, batch_size, shuffle=False, device=None):
    cuda = device is not None and torch.device(device).type == "cuda"
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                      num_workers=0, pin_memory=cuda, drop_last=False)


def ordered_loader(loader, cfg):
    return DataLoader(loader.dataset, batch_size=cfg.batch_size, shuffle=False,
                      num_workers=0, pin_memory=cfg.device.type == "cuda",
                      drop_last=False, collate_fn=loader.collate_fn)


# Converts dataset from 10 labels, into a binary classification dataset for given
# 1-vs-all tree.
class BinaryClassDataset(Dataset):
    def __init__(self, base_dataset, class_index):
        self.base_dataset = base_dataset
        self.class_index = class_index
        self.labels = (dataset_labels(base_dataset) == class_index).long().cpu()

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, index):
        image, _ = self.base_dataset[index]
        return image, self.labels[index]

In [ ]:
# Class used to create the subset dataset for the child nodes with the new labels (derived according to the tree network table).
class ChildDataset(Dataset):
    def __init__(self, parent_dataset, indices, labels):
        self.parent_dataset = parent_dataset
        self.indices = indices.tolist()
        self.labels = labels.long().cpu()

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        sample = self.parent_dataset[self.indices[index]]
        if isinstance(sample, dict):
            x = sample["pixel_values"]
        else:
            x, _ = sample
        return x, self.labels[index]

In [ ]:
def dataset_labels(dataset):
    if isinstance(dataset, TensorDataset):
        return dataset.tensors[1]
    if isinstance(dataset, (BinaryClassDataset, ChildDataset)):
        return dataset.labels
    raise TypeError("Expected TensorDataset, BinaryClassDataset, or ChildDataset")

pixel_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5] * 3, std=[0.5] * 3),
])

cifar_augmentation = transforms.Compose([
    transforms.RandomCrop(
        size=32,
        padding=4,
        padding_mode="reflect",
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0))
    ], p=0.2),
])

def augment_cifar_batch(x):
    if x.shape[0] == 0:
        return x.clone()

    return cifar_augmentation(x)

# We are caching the original images as tenors in the cpu ram to enable faster data retrieval.
# Augmentation happens on top of this cache later on.
def cache_cifar_images(hf_split, batch_size=128):
    images = torch.empty((len(hf_split), 3, 32, 32), dtype=torch.float32)
    labels = torch.empty(len(hf_split), dtype=torch.long)
    for start in range(0, len(hf_split), batch_size):
        batch = hf_split[start:start + batch_size]
        x = torch.stack([pixel_transform(image.convert("RGB")) for image in batch["img"]])
        end = start + len(x)
        images[start:end].copy_(x)
        labels[start:end] = torch.tensor(batch["label"], dtype=torch.long)
    return TensorDataset(images, labels)

In [ ]:
class TreeCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
        )
        self.feature_dim = 128 * 2 * 2

    def forward(self, x):
        return self.features(x)


class TreeNode(nn.Module):
    def __init__(self, feature_dim):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(feature_dim, 128), nn.ReLU(), nn.Linear(128, 1),
        )
        self.left_branch = None
        self.right_branch = None
        self.left_wgt = nn.Parameter(torch.tensor(0.0))
        self.right_wgt = nn.Parameter(torch.tensor(0.0))

    def forward(self, features):
        score = self.head(features).squeeze(-1)

        with torch.no_grad():
            left = ((self.left_branch(features.detach()) >= 0).to(features.dtype)
                    if self.left_branch is not None else None)
            right = ((self.right_branch(features.detach()) >= 0).to(features.dtype)
                     if self.right_branch is not None else None)
        if left is not None:
            score = score + self.left_wgt.clamp_min(0) * left
        if right is not None:
            score = score - self.right_wgt.clamp_min(0) * right
        return score

    def topology(self):
        return {
            'left': self.left_branch.topology() if self.left_branch is not None else None,
            'right': self.right_branch.topology() if self.right_branch is not None else None,
        }

    @classmethod
    def from_topology(cls, feature_dim, topology):
        node = cls(feature_dim)
        if topology['left'] is not None:
            node.left_branch = cls.from_topology(feature_dim, topology['left'])
        if topology['right'] is not None:
            node.right_branch = cls.from_topology(feature_dim, topology['right'])
        return node


class PerTreeCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = TreeCNN()
        self.root = TreeNode(self.cnn.feature_dim)
        self.cnn_frozen = False

    def freeze_cnn(self):
        self.cnn_frozen = True
        self.cnn.requires_grad_(False)
        for parameter in self.cnn.parameters():
            parameter.grad = None
        self.cnn.eval()

    def train(self, mode=True):
        super().train(mode)
        if self.cnn_frozen:
            self.cnn.eval()
        return self

    def forward(self, images, augment=False):
        images = augment_cifar_batch(images) if augment else images
        return self.root(self.cnn(images))


In [ ]:
@torch.no_grad()
def evaluate(node, loader, cfg, cnn, pos_weight=None):
    node.eval()
    cnn.eval()
    count = 0
    totals = torch.zeros(4, dtype=torch.long, device=cfg.device)
    loss_sum = torch.zeros((), device=cfg.device)
    for x, y in loader:
        x = x.to(cfg.device, non_blocking=True)
        y = y.to(cfg.device, dtype=torch.float32, non_blocking=True).reshape(-1)
        logits = node(cnn(x))
        count += y.numel()
        pred = logits >= 0
        target = y.bool()
        totals += torch.stack([(pred != target).sum(), (pred & target).sum(),
                               (pred & ~target).sum(), (~pred & target).sum()])
        loss_sum += F.binary_cross_entropy_with_logits(
            logits, y, pos_weight=pos_weight, reduction="sum")

    errors, true_positive, false_positive, false_negative = totals.tolist()
    return {"errors": errors, "accuracy": 1-errors/count,
            "loss": loss_sum.item()/count, "count": count,
            "precision": true_positive / max(true_positive + false_positive, 1),
            "recall": true_positive / max(true_positive + false_negative, 1)}


In [ ]:
def fit_node(node, dataloader, cfg, cnn, name="", epochs=None, train_cnn=False):
    node.to(cfg.device)
    cnn.to(cfg.device)
    node.head.requires_grad_(True)
    node.left_wgt.requires_grad_(True)
    node.right_wgt.requires_grad_(True)
    for branch in (node.left_branch, node.right_branch):
        if branch is not None:
            branch.requires_grad_(False)
            branch.eval()

    params = list(node.head.parameters()) + [node.left_wgt, node.right_wgt]
    if train_cnn:
        params += list(cnn.parameters())
    optimizer = torch.optim.Adam(params, lr=cfg.lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=2,
        threshold=cfg.min_delta,
        threshold_mode="abs",
        min_lr=5*1e-4,
    )
    scan = ordered_loader(dataloader, cfg)
    labels = dataset_labels(dataloader.dataset)
    total = labels.numel()
    pos = int((labels == 1).sum().item())
    pos_weight = torch.tensor(max(total-pos, 1) / max(pos, 1),
                              dtype=torch.float32, device=cfg.device)
    with torch.no_grad():
        node.left_wgt.clamp_(min=0)
        node.right_wgt.clamp_(min=0)

    def snapshot():
        return (copy.deepcopy(cnn.state_dict()) if train_cnn else None,
                copy.deepcopy(node.head.state_dict()),
                node.left_wgt.detach().clone(), node.right_wgt.detach().clone())

    initial = evaluate(node, scan, cfg, cnn, pos_weight=pos_weight)
    best_loss = initial["loss"]
    scheduler.step(best_loss)
    patience_reference = best_loss
    best_state = snapshot()
    early_stop = 0
    epoch_limit = cfg.epochs if epochs is None else epochs
    for epoch in range(epoch_limit):
        cnn.train(train_cnn)
        node.head.train()
        for x, y in dataloader:
            x = x.to(cfg.device, non_blocking=True)
            y = y.to(cfg.device, dtype=torch.float32, non_blocking=True).reshape(-1)
            x = augment_cifar_batch(x) if cfg.augment_train else x
            optimizer.zero_grad(set_to_none=True)
            with torch.set_grad_enabled(train_cnn):
                features = cnn(x)
            logits = node(features)
            loss = F.binary_cross_entropy_with_logits(logits, y, pos_weight=pos_weight)
            loss.backward()
            optimizer.step()
            with torch.no_grad():
                node.left_wgt.clamp_(min=0)
                node.right_wgt.clamp_(min=0)

        if (epoch + 1) % cfg.eval_every != 0 and epoch + 1 != epoch_limit:
            continue
        metrics = evaluate(node, scan, cfg, cnn, pos_weight=pos_weight)
        current_loss = metrics["loss"]
        if current_loss < best_loss:
            best_loss = current_loss
            best_state = snapshot()
        if current_loss < patience_reference - cfg.min_delta:
            patience_reference = current_loss
            early_stop = 0
        else:
            early_stop += 1

        previous_lr = optimizer.param_groups[0]["lr"]
        scheduler.step(current_loss)
        current_lr = optimizer.param_groups[0]["lr"]
        
        if current_lr < previous_lr:
            early_stop = 0
            print(f"{name}: learning rate {previous_lr:.2e} -> {current_lr:.2e}")

        print(f"{name} epoch {epoch+1}: accuracy={metrics['accuracy']:.2%}, "
              f"errors={metrics['errors']}, weighted BCE={current_loss:.4f}, "
              f"patience={early_stop}/{cfg.patience}")
        if early_stop >= cfg.patience and epoch + 1 >= cfg.min_epochs:
            print(f"{name}: early stopping; best weighted BCE={best_loss:.6f}")
            break

    if train_cnn:
        cnn.load_state_dict(best_state[0])
    node.head.load_state_dict(best_state[1])
    with torch.no_grad():
        node.left_wgt.copy_(best_state[2])
        node.right_wgt.copy_(best_state[3])
    cnn.eval()
    return node.eval()


In [ ]:
@torch.no_grad()
def make_child_loaders(node, parent_loader, cfg, cnn):
    scan_loader = ordered_loader(parent_loader, cfg)

    node.eval()
    cnn.eval()

    predictions, labels = [], []
    for x, target in scan_loader:
        score = node(cnn(x.to(cfg.device))).reshape(-1)
        predictions.append((score >= 0).cpu())
        labels.append(target.reshape(-1).cpu().bool())

    pred = torch.cat(predictions)
    y = torch.cat(labels)

    false_negative = ~pred & y
    false_positive = pred & ~y

    left_mask = ~(pred & y)
    right_mask = pred | y

    def build_loader(mask, targets):
        indices = mask.nonzero(as_tuple=True)[0]

        child_dataset = ChildDataset(
            parent_dataset=parent_loader.dataset,
            indices=indices,
            labels=targets[indices],
        )

        return make_loader(child_dataset, cfg.batch_size, shuffle=True,
                           device=cfg.device)

    left_loader = (
        build_loader(left_mask, false_negative)
        if false_negative.any()
        else None
    )

    right_loader = (
        build_loader(right_mask, false_positive)
        if false_positive.any()
        else None
    )

    return left_loader, right_loader

In [ ]:
def fit_tree(dataloader, cfg, name="root"):
    tree = PerTreeCNN().to(cfg.device)
    cnn = tree.cnn
    frontier = [(tree.root, dataloader, name)]
    node_records = [(tree.root, dataloader, name)]

    for depth in range(cfg.max_depth):
        if not frontier:
            break
        print(f"{name}: feature learning at depth {depth}")
        for node, loader, node_name in frontier:
            fit_node(node, loader, cfg, cnn, name=node_name + " grow", train_cnn=True)

        next_frontier = []
        for node, loader, node_name in frontier:
            left_loader, right_loader = make_child_loaders(node, loader, cfg, cnn)
            for side, child_loader in (("left", left_loader), ("right", right_loader)):
                if child_loader is not None:
                    child = TreeNode(cnn.feature_dim).to(cfg.device)
                    setattr(node, side + "_branch", child)
                    next_frontier.append((child, child_loader, node_name + "-" + side))
                    node_records.append((child, child_loader, node_name + "-" + side))
        frontier = next_frontier

    tree.freeze_cnn()
    print(f"{name}: CNN frozen; fitting leaves, then refitting upward")

    for node, loader, node_name in reversed(node_records):
        has_children = (
            node.left_branch is not None
            or node.right_branch is not None
        )

        fit_node( node,loader,cfg,cnn,name=node_name + (" refit" if has_children else " leaf"),
                 epochs=cfg.refit_epochs if has_children else cfg.epochs, train_cnn=False )
        node.requires_grad_(False)

    tree.requires_grad_(False)
    return tree.eval()


In [ ]:
class OneVsAllTrees(nn.Module):
    def __init__(self, trees):
        super().__init__()
        if len(trees) != 10:
            raise ValueError("Expected exactly ten class-index-ordered trees")
        self.trees = nn.ModuleList(trees)

    def forward(self, images):
        return torch.stack([tree(images) for tree in self.trees], dim=1)

    @torch.no_grad()
    def predict(self, images):
        return self(images).argmax(dim=1)

@torch.no_grad()
def evaluate_multiclass(model, loader, cfg):
    model.eval()
    classes = torch.arange(10, device=cfg.device)
    confusion = torch.zeros(10, 10, dtype=torch.long)
    binary_tp = torch.zeros(10, dtype=torch.long)
    binary_fp = torch.zeros(10, dtype=torch.long)
    binary_fn = torch.zeros(10, dtype=torch.long)
    binary_errors = torch.zeros(10, dtype=torch.long)
    loss_sum = torch.zeros(10, dtype=torch.float64)
    count = zero_positive = multiple_positive = 0
    for x, y in loader:
        x = x.to(cfg.device, non_blocking=True)
        y = y.to(cfg.device, dtype=torch.long, non_blocking=True)
        scores = model(x)
        target = y[:, None] == classes[None, :]
        binary_pred = scores >= 0
        pred = scores.argmax(dim=1)
        confusion += torch.bincount((y * 10 + pred).cpu(), minlength=100).reshape(10, 10)
        binary_tp += (binary_pred & target).sum(0).cpu()
        binary_fp += (binary_pred & ~target).sum(0).cpu()
        binary_fn += (~binary_pred & target).sum(0).cpu()
        binary_errors += (binary_pred != target).sum(0).cpu()
        loss_sum += F.binary_cross_entropy_with_logits(
            scores, target.to(scores.dtype), reduction="none").sum(0).cpu().double()
        n_positive = binary_pred.sum(1)
        zero_positive += int((n_positive == 0).sum().item())
        multiple_positive += int((n_positive > 1).sum().item())
        count += len(y)
    if count == 0:
        raise ValueError("Empty evaluation loader")
    return {
        "count": count,
        "accuracy": confusion.diag().sum().item() / count,
        "errors": count - confusion.diag().sum().item(),
        "confusion_matrix": confusion,
        "binary_accuracy": 1 - binary_errors.double() / count,
        "binary_bce": loss_sum / count,
        "binary_precision": binary_tp.double() / (binary_tp + binary_fp).clamp_min(1),
        "binary_recall": binary_tp.double() / (binary_tp + binary_fn).clamp_min(1),
        "argmax_class_recall": confusion.diag().double() / confusion.sum(1).clamp_min(1),
        "zero_positive_trees": zero_positive,
        "multiple_positive_trees": multiple_positive,
    }


def report_metrics(title, metrics, class_names):
    print(f"{title}: accuracy={metrics['accuracy']:.2%}, "
          f"errors={metrics['errors']}/{metrics['count']}")
    print("class         binary acc  binary BCE  precision  recall  argmax recall")
    for i, name in enumerate(class_names):
        print(f"{name:12} {metrics['binary_accuracy'][i]:10.2%} "
              f"{metrics['binary_bce'][i]:11.4f} "
              f"{metrics['binary_precision'][i]:10.2%} "
              f"{metrics['binary_recall'][i]:7.2%} "
              f"{metrics['argmax_class_recall'][i]:13.2%}")
    print("Examples with zero positive trees:", metrics['zero_positive_trees'])
    print("Examples with multiple positive trees:", metrics['multiple_positive_trees'])

In [ ]:
def fit_one_vs_all(train_dataset, cfg, class_names):
    trees = []
    for class_index, class_name in enumerate(class_names):
        print(f"\nClass {class_index}: {class_name} versus all")
        binary_dataset = BinaryClassDataset(train_dataset, class_index)
        loader = make_loader(binary_dataset,cfg.batch_size,shuffle=True,device=cfg.device)
        tree = fit_tree(loader,cfg,name=f"{class_index}:{class_name}")
        trees.append(tree)
    return OneVsAllTrees(trees).to(cfg.device).eval()

In [ ]:
dataset = load_dataset("uoft-cs/cifar10")
assert len(dataset["train"]) == 50_000
assert len(dataset["test"]) == 10_000
class_names = dataset["train"].features["label"].names
assert len(class_names) == 10

train_dataset = cache_cifar_images(dataset["train"])
test_dataset = cache_cifar_images(dataset["test"])
train_loader = make_loader(train_dataset, cfg.batch_size, device=cfg.device)
test_loader = make_loader(test_dataset, cfg.batch_size, device=cfg.device)
print("Classes:", class_names)
print("Cached training images:", train_dataset.tensors[0].shape)
print("Cached test images:", test_dataset.tensors[0].shape)
print("Train class counts:", torch.bincount(train_dataset.tensors[1], minlength=10))
print("Test class counts:", torch.bincount(test_dataset.tensors[1], minlength=10))

In [ ]:
model = fit_one_vs_all(train_dataset, cfg, class_names)

node_counts = [sum(isinstance(m, TreeNode) for m in tree.modules()) for tree in model.trees]
cnn_counts = [sum(isinstance(m, TreeCNN) for m in tree.modules()) for tree in model.trees]
assert cnn_counts == [1] * 10
print("Nodes per tree:", dict(zip(class_names, node_counts)))
print("CNN blocks per tree:", cnn_counts)
print("Total nodes:", sum(node_counts))
print("Inference ensemble parameters:", sum(p.numel() for p in model.parameters()))
print("Trainable parameters after all refits:",
      sum(p.numel() for p in model.parameters() if p.requires_grad))


In [ ]:
train_metrics = evaluate_multiclass(model, train_loader, cfg)
test_metrics = evaluate_multiclass(model, test_loader, cfg)
report_metrics("Training", train_metrics, class_names)
print()
report_metrics("Official test", test_metrics, class_names)

fig, ax = plt.subplots(figsize=(9, 8))
confusion = test_metrics["confusion_matrix"].numpy()
im = ax.imshow(confusion, cmap="Blues")
ax.set_xticks(range(10), class_names, rotation=45, ha="right")
ax.set_yticks(range(10), class_names)
ax.set_xlabel("Predicted class (argmax)")
ax.set_ylabel("True class")
ax.set_title("CIFAR-10 official test confusion matrix")
for i in range(10):
    for j in range(10):
        ax.text(j, i, str(confusion[i, j]), ha="center", va="center",
                color="white" if confusion[i, j] > confusion.max() / 2 else "black",
                fontsize=8)
fig.colorbar(im, ax=ax)
fig.tight_layout()
plt.show()

In [ ]:
param_counts = torch.tensor(
    [sum(p.numel() for p in tree.parameters()) for tree in model.trees],
    dtype=torch.float64,
)
cnn_counts = [sum(p.numel() for p in tree.cnn.parameters()) for tree in model.trees]
head_counts = [sum(p.numel() for p in tree.root.parameters()) for tree in model.trees]
print("CNN parameters per tree:", cnn_counts)
print("Node-head and correction parameters per tree:", head_counts)
print("Total parameters per tree:", param_counts.long().tolist())
print(f"Mean: {param_counts.mean().item():,.2f}")
print(f"Std:  {param_counts.std(unbiased=False).item():,.2f}")
